<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l5.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L5 · Correlación y beta
¿Se mueven juntos BTC, ETH y SOL? Matriz de correlación, mapa de calor y betas contra BTC.

In [ ]:
import pandas as pd
from pathlib import Path

CSV = 'c1_l5_btc_eth_sol.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            df = pd.read_csv(cand)
            break
    print('Fuente: local')
cols = ["btc_ret", "eth_ret", "sol_ret"]
print(df.shape)
print(df.head())

## Matriz de correlación
De −1 a +1. BTC–ETH cerca de 0.9 son casi gemelos: tener ambos apenas diversifica.

In [ ]:
corr = df[cols].corr()
print(corr.round(2))

## Mapa de calor
El bloque más intenso marca la pareja más sincronizada.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.matshow(corr.values, cmap="YlGn")
ax.set_xticks(range(3), cols)
ax.set_yticks(range(3), cols)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im)
fig.tight_layout()
plt.show()

## Beta contra BTC
Sensibilidad al mercado: covarianza con BTC entre varianza de BTC. Beta < 1 se mueve menos que el jefe, en ambas direcciones.

In [ ]:
var_m = df["btc_ret"].var(ddof=0)
for c in ["eth_ret", "sol_ret"]:
    beta = df[[c, "btc_ret"]].cov().iloc[0, 1] / df["btc_ret"].var()
    print(f"beta {c}: {beta:.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["btc_ret"], df["sol_ret"], alpha=0.6, color="#5eead4")
ax.set_xlabel("BTC retorno (%)")
ax.set_ylabel("SOL retorno (%)")
ax.set_title("SOL vs BTC: la pendiente es la beta")
fig.tight_layout()
plt.show()

In [ ]:
# Chequeos automáticos
assert len(df) == 60, "se esperan 60 días"
assert abs(corr.loc["btc_ret", "eth_ret"] - corr.loc["eth_ret", "btc_ret"]) < 1e-12, "la matriz es simétrica"
assert all(abs(corr.loc[c, c] - 1) < 1e-12 for c in cols), "la diagonal es 1"
assert 0.5 < corr.loc["btc_ret", "eth_ret"] < 1, "BTC–ETH alta pero no perfecta"
print("OK: correlaciones y betas verificadas")